# A little Wonder

Let's take Muse Wonder snacking. Have `MODEL_API_KEY` available in the notebook's environment, then run the cells you want to try. Calls use Meta's Model API.


## Hello, Popsicle

Same Chat, different model. The model name supplies the connection defaults.


In [ ]:
from chatsnack import Chat

popsicle = Chat("Respond only with the word POPSICLE.", model="muse-wonder-1.0")
popsicle.ask("What's for dessert?")


## Yummy YAML

Keep a prompt we can fill in for another occasion.


In [ ]:
menu = (
    Chat(name="WonderMenu", model="muse-wonder-1.0")
    .system("Invent cheerful, crunchy snacks.")
    .user("Recommend one snack for {occasion}.")
)
print(menu.yaml)
menu.save()


A minimal hand-authored version of this prompt:

```yaml
params:
  model: muse-wonder-1.0
messages:
  - system: Invent cheerful, crunchy snacks.
  - user: Recommend one snack for {occasion}.
```

The saved Chat also keeps its resolved connection settings. Load it by name and supply the filling:


In [ ]:
reused = Chat(name="WonderMenu")
reused.load()
print(reused.ask(occasion="a moon picnic"))


## Keep snacking

`ask()` gives us an answer. `chat()` gives us a new Chat with the answer included, ready to continue.


In [ ]:
thread = reused.chat(occasion="a moon picnic")
thread = thread.chat("Give that snack a two-word name.")
print(thread.last)

thread.name = "MoonPicnicThread"
thread.save()


## Bring a utensil

A pantry check is an ordinary Python function. Give it to a Chat with `utensils=[...]`.


In [ ]:
from chatsnack import utensil

@utensil
def stock(snack: str) -> dict:
    """Check the demo pantry."""
    return {"snack": snack, "available": 7}

pantry = (
    Chat(name="WonderPantry", model="muse-wonder-1.0", utensils=[stock])
    .system("Use stock to check what we can pack for our picnic.")
    .user("Check {snack}.")
)
pantry.reasoning.effort = "minimal"
print(pantry.yaml)
pantry.save()


Reload the prompt with its Python capability attached. The filling stays live until we ask:


In [ ]:
reused_pantry = Chat(name="WonderPantry", utensils=[stock])
reused_pantry.load()
stock_thread = reused_pantry.chat(snack="popcorn")
print(stock_thread.last)


## One bite at a time

A copy keeps the conversation's provider binding. Turn on streaming and read the answer as it arrives:


In [ ]:
stream_chat = stock_thread.copy()
stream_chat.stream = True
for chunk in stream_chat.listen("How much popcorn was available? Use our history."):
    print(chunk, end="")


Close the clients when we're done:


In [ ]:
for chat in (popsicle, menu, reused, pantry, reused_pantry, stream_chat):
    await chat.close_a()


Explicit connection settings still win. See the [provider guide](../docs/guides/providers.md#muse-wonder) for overrides and compatibility notes.
